**The Problem**
*A bookstore assistant*. The model handles a customer's questions about stock and pricing, but the shop's data lives in your Python — the model has no way to know any of it.

Three tools, kept deliberately tiny so nothing distracts from the mechanics:

```
check_stock(title: str) -> str              # is it available?
get_price(title: str) -> float              # what does it cost?
apply_member_discount(price: float) -> float # 15% off
```

Backed by a hardcoded catalogue — say Dune ($18.99, 12 copies), Neuromancer ($15.50, out of stock), Snow Crash ($16.75, 3 copies).

Why this problem and not something the model could answer itself: it satisfies all three conditions from Concept 1. The model can't see the catalogue, can't compute the discount without the price, and the system instruction will forbid it from guessing. Ask it "is Dune in stock" with no tools and it can only make something up.

Three questions the notebook has to answer

Each one exercises a different mechanic, and they build:

|Question|	What it forces|
|-|-|
|"Do you have Dune?"|one call → one result → one answer|
|"Do you have Dune or Neuromancer?"	|two calls in one turn — parallel|
|"What would I pay for Dune as a member?"|	get_price → then apply_member_discount on its output — sequential|

That third one is the important one. apply_member_discount needs a price the model doesn't have until get_price returns, so it cannot batch them. That's what makes the turn loop necessary rather than optional, and it's a genuine dependency — unlike the notebook's laptop example, where the customer supplied the price themselves.

Cell plan:
1. Setup — imports, client, MODEL_ID
2.  The three tools + local tests (no LLM yet)
3.  The registry
4.  Q1: catch the request         (Concept 3a)
5.  Q1: execute it                (Concept 3b)
6.  Q1: send the result back      (Concept 3c)
7.  Q2: two calls, loop + zip     (Concept 3d)
8.  Q3: the turn loop             (Concept 3e)
9.  The same thing automatically  (Concept 1, revisited)

In [18]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Load API credentials securely
load_dotenv(override=True)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
print(GOOGLE_API_KEY)

# Initialize the Gemini Client
client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL_ID = "gemini-3.5-flash-lite"

AQ.Ab8RN6L_FrJiX1PMjn7VZWJmSnGFEfTB-Ac5TJL83MW-It_KOw


In [19]:
# title -> (price, availability)
CATALOGUE = {
    "DUNE": (18.99, "12 copies"),
    "NEUROMANCER": (15.50, "out of stock"),
    "SNOW CRASH": (16.75, "3 copies"),
}


def check_stock(title: str) -> str:
    """Checks whether a book is currently available in the shop.

    Use this when the customer asks about availability or copies on hand.
    Returns the number of copies, "out of stock", or "unavailable" if we
    do not carry the title. Does not return a price.
    """
    print(f"[tool] check_stock({title!r})")
    return CATALOGUE.get(title.strip().upper(), ("", "unavailable"))[1]


def get_price(title: str) -> float:
    """Looks up the list price of a book in dollars.

    Use this when the customer asks what a book costs. Returns None if we
    do not carry the title. Does not apply any discount.
    """
    print(f"[tool] get_price({title!r})")
    entry = CATALOGUE.get(title.strip().upper())
    return entry[0] if entry else None


def apply_member_discount(price: float) -> float:
    """Applies the 15 percent member discount to a list price.

    Requires a price in dollars, which you must obtain from get_price first.
    Returns the final amount the member pays.
    """
    print(f"[tool] apply_member_discount({price!r})")
    if price is None:
        return None
    return round(price * 0.85, 2)



In [20]:
print(check_stock("dune"))            # 12 copies
print(check_stock("  Snow Crash  "))  # 3 copies
print(check_stock("Dracula"))         # unavailable

print(get_price("NEUROMANCER"))       # 15.5
print(get_price("Dracula"))           # None

print(apply_member_discount(18.99))   # 16.14
print(apply_member_discount(None))    # None

[tool] check_stock('dune')
12 copies
[tool] check_stock('  Snow Crash  ')
3 copies
[tool] check_stock('Dracula')
unavailable
[tool] get_price('NEUROMANCER')
15.5
[tool] get_price('Dracula')
None
[tool] apply_member_discount(18.99)
16.14
[tool] apply_member_discount(None)
None


In [21]:
TOOL_REGISTRY = {
    "check_stock": check_stock,
    "get_price": get_price,
    "apply_member_discount": apply_member_discount,
}

TOOLS = list(TOOL_REGISTRY.values())
name = "get_price"
fn = TOOL_REGISTRY[name]
fn(title="Dune")          # 18.99

[tool] get_price('Dune')


18.99

In [22]:
SYSTEM_INSTRUCTION = (
    "You are a bookshop assistant. You must use the provided tools to answer "
    "questions about availability and pricing. Never guess or answer from memory."
)

q1 = "Do you have Dune?"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=q1,
    config=types.GenerateContentConfig(
        tools=TOOLS,
        automatic_function_calling={"disable": True},
        system_instruction=SYSTEM_INSTRUCTION,
        temperature=0.0,
    ),
)

# if response.function_calls:
#     ...
# else:
#     print(response.text)

print("text: ", response.text)
print("calls:", response.function_calls)


text:  None
calls: [FunctionCall(
  args={
    'title': 'Dune'
  },
  id='call_2252616',
  name='check_stock'
)]


In [23]:
for part in response.candidates[0].content.parts:
    print(part)

media_resolution=None code_execution_result=None executable_code=None file_data=None function_call=FunctionCall(
  args={
    'title': 'Dune'
  },
  id='call_2252616',
  name='check_stock'
) function_response=None inline_data=None text=None thought=None thought_signature=b'\x12^\n\\\x01\x11M2\x0f\xbd\x1a\x83j\x1b\xf8\x1e]`\xc6\x08\x13\x86\xa1\xa3A\xed\xa80+\xf7/[o\x84n\xdc|\xca\xf5\xef\x86\x96\x8c\x8c\xdf\xab\x80\xeb7$\xbc\xf5\x11\x9a\x00\x02[\xa7\xb5\xf9\xfa\xfe\xce\xb0\xd0&\xf64\xf2\xa7{\x80r\x1c\xf5Iy\x8d\x03\xfc\xed\x03\xf2)P\xc4\x80}fs\x1b\xdf\xa1uw\x86' video_metadata=None tool_call=None tool_response=None part_metadata=None audio_transcription=None


In [24]:
call = response.function_calls[0]

fn = TOOL_REGISTRY.get(call.name)

if fn is None:
    result = f"Error: no tool named {call.name}"
else:
    result = fn(**call.args)

print("result:", result)

[tool] check_stock('Dune')
result: 12 copies


In [ ]:
user_turn = types.Content(
    role="user",
    parts=[types.Part(text=q1)]
)

# Reuse the model's own turn object rather than rebuilding it —
# it carries thought_signature and other internal fields you can't
# reconstruct by hand.
model_turn = response.candidates[0].content

tool_turn = types.Content(
    role="user",
    parts=[types.Part(
        function_response=types.FunctionResponse(
            id=call.id,
            name=call.name,
            response={"result": result}
        )
    )]
)

final = client.models.generate_content(
    model=MODEL_ID,
    contents=[user_turn, model_turn, tool_turn],
    config=types.GenerateContentConfig(
        tools=TOOLS,
        automatic_function_calling={"disable": True},
        system_instruction=SYSTEM_INSTRUCTION,
        temperature=0.0,
    ),
)

print(final.text)

NameError: name 'local_results' is not defined